# Population & Temperature Measurements Workflow

## Description

This is one of four template notebooks that together replace the original
monolithic workflow. **This notebook covers Population & temperature experiments** for a single qubit using the LabOne Q workflow architecture:

1. [Single Shot 0 & 1 Measurements](#one-single-shot-measurement)
2. [Single Shot Measurement for Different Rabi Frequencies](#single-shot-measurement-for-different-rabi-frequencies)
3. [Single Shot Measurement for Different Integration Lengths and Delays](#single-shot-measurement-for-different-integration-lengths-and-delays)
4. [Single Shot Measurement for Different Readout Amplitudes and Lengths](#single-shot-measurement-for-different-readout-amplitudes-and-lengths)
5. [Rabi Population Measurements (RPM)](#rabi-population-measurements-rpm)'
6. [Three Level Population Measurements](#three-level-population-measurements)
7. [Population Measurements for Different Readout Frequencies](#population-measurements-for-different-readout-frequencies)

Run this notebook for single shot and temperature measurements from population distribution. The other three
template notebooks under `qubit_thermometry/templates/` load that saved QPU
and continue from there:

- Tune-up experiments.
- Fast Flux Drive.
- SINIS Calibration, Statistic and Temperature Sweep measurements.


## Changelog

All changes must be logged here! Do not make changes to this notebook other than expanding or modifying the standard experiments in a meaningful! All device-specific parameters and measurements have to be done in a different notebook!

Stick to semantic versioning:
- **Last Digit**: Fix a bug that does not add any new features and is backwards compatible -> increment last digit by one (e.g. 0.0.4 to 0.0.5)

- **Middle Digit**: Add a new feature while keeping backwards compatibility (older versions can still be executed without any modifications) -> increment middle digit by one and set last digit to 0 (e.g. 1.2.12 to 1.3.0)

- **First Digit**: Make changes that will break older versions, making them incompatible with the new version -> increment first digit by one and set all other digits to 0 (e.g. 3.2.9 to 4.0.0)

**Current Version:** <font color="#008000">**v2.0.0**</font>

**2026-09-27**, *Kha*:  Updating from version v1.7.3, separated specified script for different experiments. Single shot experiments and temperature measurement
                        population distribution using Boltzmann equation.                                                                                           (v2.0.0)

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/Users/khatran/Desktop/qubit-thermometry_v2.0")
sys.path.insert(0, str(REPO_ROOT / "qubit_thermometry" / "src"))

In [ ]:
from copy import deepcopy
from functools import partial
from pprint import pprint

import attrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from laboneq.simple import dsl
from laboneq_applications.experiments import iq_blobs

from qubit_thermometry.helper import data_io, plotting, qpu_io, setup, single_shot, temperature
from qubit_thermometry.workflow.analysis import population as population_analysis
from qubit_thermometry.workflow.analysis import rabi_population as rabi_population_analysis
from qubit_thermometry.workflow.analysis import single_shot as single_shot_analysis
from qubit_thermometry.workflow.experiments.population import population_experiment_workflow
from qubit_thermometry.workflow.experiments.quick_rabi_population import (
    quick_rabi_population_experiment_workflow,
)
from qubit_thermometry.workflow.experiments.rabi_population import (
    rabi_population_experiment_workflow,
)

`DeviceSetup` (physical instrument connections/ports/IPs) is a separate concern from the QPU (qubit calibration parameters) and is never saved/loaded -- it is cheap and deterministic to rebuild, so it is re-entered here just like the sample/qubit/cooldown identifiers below. The qubits returned by `build_device_setup` are fresh/default ones and are discarded; the actual tuned qubits come from the QPU saved by the tune-up notebook.

In [ ]:
device_setup, _default_qubits = setup.build_device_setup(
    shfqc_device_id="",  # TODO
    number_of_qubits=1,  # TODO
    ip_address="localhost",  # TODO
)
session = setup.connect_session(device_setup)

Sample/qubit/cooldown identifiers, used to locate the data directory and the tuned QPU saved by the tune-up notebook.

In [ ]:
sample_name = ""  # TODO
qubit_name = ""  # TODO
cooldown_start_date = ""  # TODO

In [ ]:
data_root_directory, qpu_file_path = setup.init_data_saving(
    sample_name, cooldown_start_date, qubit_name
)
get_path_to_file = partial(setup.get_path_to_file, directory=data_root_directory)

qpu = qpu_io.load_qpu(qpu_file_path)
qubits = qpu.quantum_elements
qops = qpu.quantum_operations

folder_store, logging_store = setup.setup_logbook(data_root_directory)

# Single Shot 0 and 1 Measurements
Single-shot (unaveraged) readout of the qubit prepared in g, e (and f).

1. **One Single Shot Measurement** — acquire the single shots for each prepared state, classify them with a linear discriminant, and quantify the readout with the state distance, the relative shot noise and the correct-state assignment fidelity.
2. **Single Shot vs. Drive Pulse Length** — repeat the measurement for a range of pi-pulse lengths (i.e. Rabi frequencies) and keep the length that *maximizes* the g/e/f assignment fidelity.
3. **Single Shot vs. Integration Length and Delay** — repeat the measurement for a range of integration delays and integration lengths at a fixed readout pulse and keep the combination that *maximizes* the assignment fidelity.
4. **Single Shot vs. Readout Amplitude and Length** — repeat the measurement for a range of readout amplitudes and readout lengths and keep the combination that *maximizes* the assignment fidelity.

At every sweep point, the optimum is the sweep point with the highest **g/e/f correct-state assignment fidelity** returned by the `iq_blobs` analysis workflow (the average of the diagonal of the 3x3 assignment/confusion matrix). `rel_std_0` (the relative standard deviation of the projected state-0 shots) is still computed and plotted for reference, but is no longer used to pick the optimum.

The readout parameters optimized during Readout Optimization are the starting point here; the best settings found below are written back into the QPU.

## One Single Shot Measurement

#### Experiment Parameters

In [ ]:
update_global_parameters = True

states = "gef"
n_avg_exponent = 17

readout_integration_length = 1.25e-6

qubit_to_measure = qubits[0]

temp_pars = deepcopy(qubit_to_measure.parameters)
temp_pars.readout_integration_length = readout_integration_length
temporary_parameters = {qubit_to_measure.uid: temp_pars}

print("Readout length:             ", temp_pars.readout_length)
print("Readout integration length: ", temp_pars.readout_integration_length)
print("Readout integration delay:  ", temp_pars.readout_integration_delay)
print("Readout amplitude:          ", temp_pars.readout_amplitude)
print("Reset delay length:         ", temp_pars.reset_delay_length)
print("Readout resonator frequency:", temp_pars.readout_resonator_frequency)
print("Number of shots per state:  ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = iq_blobs.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(False)
# the analysis classifies the shots with sklearn LinearDiscriminantAnalysis
options.do_analysis(True)

exp_workflow = iq_blobs.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[qubit_to_measure.uid],
    states=states,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
result = workflow_result.output
shots_per_state = single_shot.collect_shots_from_result(result, qubit_to_measure.uid, states)

zero_data, one_data = shots_per_state["g"], shots_per_state["e"]

assignment_fidelities = workflow_result.tasks["analysis_workflow"].output
assignment_matrices = setup.get_analysis_task_output(
    workflow_result, "calculate_assignment_matrices"
)
assignment_fidelity = assignment_fidelities.get(qubit_to_measure.uid)

In [ ]:
fig, ax = single_shot_analysis.plot_iq_scatter(shots_per_state, qubit_to_measure.uid, states)
fig.savefig(get_path_to_file("Single_shot_points_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
res_ss = single_shot.analyze_single_shots(shots_per_state, state_0="g", state_1="e", plot=True)
single_shot.summarize_single_shot_metrics(res_ss, assignment_fidelity)
pprint({k: v for k, v in res_ss.items() if not isinstance(v, np.ndarray) and k != "figure"})

In [ ]:
fig, axs = single_shot_analysis.plot_iq_histograms(shots_per_state, qubit_to_measure.uid, states)
fig.savefig(get_path_to_file("Single_shot_hist_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
zero_data_proj = res_ss["shots_0_proj"]
one_data_proj = res_ss["shots_1_proj"]
distance = res_ss["distance"]

single_shot_analysis.plot_projected_distance_histogram(
    zero_data_proj, one_data_proj, distance, qubit_to_measure.uid
)

In [ ]:
# single Gaussian fit of the projected state-0 shots
single_shot_analysis.fit_and_plot_gaussian(zero_data_proj / distance, qubit_to_measure.uid, state_label="0")

In [ ]:
# bimodal fit of the projected state-1 shots -> residual ground population
single_shot_analysis.fit_and_plot_bimodal(
    one_data_proj / distance,
    expected=(-1, 0.2, 5e3 * 0.5, 1, 0.2, 5e3 * 0.5),
    qubit_uid=qubit_to_measure.uid,
    fit_label="state 1",
    color="red",
    zero_first=True,
)

In [ ]:
# bimodal fit of the projected state-0 shots -> residual excited population
single_shot_analysis.fit_and_plot_bimodal(
    zero_data_proj / distance,
    expected=(-1.5, 1.0, 5e3 * 0.5, 1, 1.0, 5e3 * 0.5),
    qubit_uid=qubit_to_measure.uid,
    fit_label="state 0",
    color="blue",
    zero_first=False,
)

In [ ]:
single_shot_analysis.plot_iq_density(zero_data, one_data, qubit_to_measure.uid)

In [ ]:
# the assignment matrix produced by the iq_blobs analysis
single_shot_analysis.print_assignment_matrix(
    assignment_matrices.get(qubit_to_measure.uid), assignment_fidelity
)

#### Update Parameters

In [ ]:
data_ss = {f"shots_{s}": shots_per_state[s] for s in states}
data_ss["states"] = list(states)
data_ss["shot_index"] = np.arange(shots_per_state["g"].shape[-1])
data_ss["n_shots"] = 2**n_avg_exponent
data_ss.update({k: v for k, v in res_ss.items() if isinstance(v, (int, float, complex))})
if qubit_to_measure.uid in assignment_matrices:
    data_ss["assignment_matrix"] = assignment_matrices[qubit_to_measure.uid]
    data_ss["assignment_fidelity"] = assignment_fidelity

payload = data_io.build_mat_payload(
    temporary_parameters[qubit_to_measure.uid], sample_name, qubit_name, cooldown_start_date,
    data=data_ss,
)
data_io.save_mat(get_path_to_file("Single_shots_all_unsh_", ".mat"), payload)

In [ ]:
if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("Readout amplitude: ", qubit_to_measure.parameters.readout_amplitude)
print("Readout length: ", qubit_to_measure.parameters.readout_length)
print("Readout integration length: ", qubit_to_measure.parameters.readout_integration_length)
print("Readout integration delay: ", qubit_to_measure.parameters.readout_integration_delay)
print("Reset delay length: ", qubit_to_measure.parameters.reset_delay_length)

## Single Shot Measurement for Different Rabi Frequencies
*Optimum: the drive length / Rabi frequency with the highest g/e/f assignment fidelity.*

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 15  # reduce: this is a sweep over 36 pi-pulse lengths
states = "gef"

drive_length_arr = np.linspace(60e-9, 400e-9, 36)
rabi_freq_arr = 1 / drive_length_arr

# scale the calibrated pi amplitude to the swept drive length instead of
# re-deriving it from a linear rabi_slope / rabi_intercept calibration.
qubit_to_measure = qubits[0]
rabi_slope = (
    qubit_to_measure.parameters.ge_drive_amplitude_pi * qubit_to_measure.parameters.ge_drive_length
)
ge_drive_amplitude_pi_arr = np.clip(rabi_slope / drive_length_arr, 0.0, 1.0)

print(pd.DataFrame({
    "ge_drive_length": drive_length_arr,
    "rabi_freq_MHz": rabi_freq_arr * 1e-6,
    "ge_drive_amplitude_pi": ge_drive_amplitude_pi_arr,
}))

#### Run Workflow

In [ ]:
options = iq_blobs.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)  # one IQ-blob figure per drive length
options.do_analysis(True)

ss_drive_length_sweep_shots = []
ss_drive_length_sweep_metrics = []
ss_drive_length_sweep_fidelities = []

with setup.logging_disabled(folder_store, logging_store):
    for drive_length, drive_amplitude_pi in zip(drive_length_arr, ge_drive_amplitude_pi_arr):
        print("Drive length:", round(drive_length * 1e9, 3), "ns",
              "| pi amplitude:", round(float(drive_amplitude_pi), 5))

        temp_pars = deepcopy(qubit_to_measure.parameters)
        temp_pars.ge_drive_length = float(drive_length)
        temp_pars.ge_drive_amplitude_pi = float(drive_amplitude_pi)
        temporary_parameters = {qubit_to_measure.uid: temp_pars}

        exp_workflow = iq_blobs.experiment_workflow(
            session=session, qpu=qpu, qubits=[qubit_to_measure.uid], states=states,
            options=options, temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()

        shots = single_shot.collect_shots_from_result(workflow_result.output, qubit_to_measure.uid, states)
        ss_drive_length_sweep_shots.append(np.array([shots[s] for s in states]))

        # diagnostic-only, g/e relative STD (kept for reference/comparison)
        res = single_shot.analyze_single_shots(shots, state_0="g", state_1="e", plot=False)
        ss_drive_length_sweep_metrics.append(
            {k: v for k, v in res.items() if isinstance(v, (int, float, complex))}
        )
        # g/e/f assignment fidelity - the metric used to pick the optimum
        ss_drive_length_sweep_fidelities.append(
            workflow_result.tasks["analysis_workflow"].output.get(qubit_to_measure.uid, np.nan)
        )

ss_drive_length_sweep_arr = np.array(ss_drive_length_sweep_shots)
print("Sweep array shape (drive lengths, states, shots):", ss_drive_length_sweep_arr.shape)

#### Analysis Plots

In [ ]:
rel_std_0_arr = np.array([m["rel_std_0"] for m in ss_drive_length_sweep_metrics])
fidelity_arr = np.array(ss_drive_length_sweep_fidelities, dtype=float)

optimal_index, optimal_fidelity = single_shot_analysis.pick_optimal_1d(fidelity_arr)
print("Best drive length:", round(drive_length_arr[optimal_index] * 1e9, 3), "ns",
      "| Rabi frequency:", round(rabi_freq_arr[optimal_index] * 1e-6, 3), "MHz",
      "| assignment fidelity:", f"{optimal_fidelity * 100:0.2f} %")

fig, ax = single_shot_analysis.plot_fidelity_vs_1d(
    rabi_freq_arr * 1e-6, fidelity_arr, optimal_index, "Rabi frequency, MHz",
    qubit_to_measure.uid, secondary=(rel_std_0_arr, "Relative STD state 0"),
)
fig.savefig(get_path_to_file("Single_shots_vs_rabi_freq_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_ss = {
    "ss_drive_length_sweep_arr": ss_drive_length_sweep_arr,
    "states": list(states),
    "ge_drive_length": drive_length_arr,
    "rabi_freq": rabi_freq_arr,
    "ge_drive_amplitude_pi": ge_drive_amplitude_pi_arr,
    "assignment_fidelity": fidelity_arr,
    "rel_std_0": rel_std_0_arr,
    "optimal_index": optimal_index,
    "n_shots": 2**n_avg_exponent,
    "comment": "Sweep of the pi-pulse length. Optimal point maximizes assignment_fidelity (g/e/f).",
}
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_ss)
data_io.save_mat(get_path_to_file("Single_shots_0_pi_diff_pipulses_", ".mat"), payload)

In [ ]:
if update_global_parameters:
    temp_pars = deepcopy(qubit_to_measure.parameters)
    temp_pars.ge_drive_length = float(drive_length_arr[optimal_index])
    temp_pars.ge_drive_amplitude_pi = float(ge_drive_amplitude_pi_arr[optimal_index])
    qubit_to_measure.parameters = temp_pars
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("ge drive length: ", qubit_to_measure.parameters.ge_drive_length)
print("ge drive amplitude pi: ", qubit_to_measure.parameters.ge_drive_amplitude_pi)

## Single Shot Measurement for Different Integration Lengths and Delays
*Optimum: the (integration length, integration delay) combination with the highest g/e/f assignment fidelity.*

#### Experiment Parameters

In [ ]:
update_global_parameters = True

n_avg_exponent = 16
states = "gef"

qubit_to_measure = qubits[0]
readout_length = qubit_to_measure.parameters.readout_length
print("Readout pulse length:", readout_length)

readout_integration_delay_arr = np.linspace(60, 200, 2) * 1e-9
readout_integration_length_arr = np.linspace(1, 2, 3) * 1e-6

print("Readout integration delays (ns): ", readout_integration_delay_arr * 1e9)
print("Readout integration lengths (us):", readout_integration_length_arr * 1e6)

#### Run Workflow

In [ ]:
options = iq_blobs.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)
options.do_analysis(True)

ss_delay_sweep_shots = []
ss_delay_sweep_metrics = []
ss_delay_sweep_fidelities = []
ss_delay_sweep_matrices = []

with setup.logging_disabled(folder_store, logging_store):
    for integration_delay in readout_integration_delay_arr:
        print("Integration delay:", round(integration_delay * 1e9, 3), "ns")

        shots_inner, metrics_inner, fidelities_inner, matrices_inner = [], [], [], []
        for integration_length in readout_integration_length_arr:
            print("  Integration length:", round(integration_length * 1e6, 3), "us")

            temp_pars = deepcopy(qubit_to_measure.parameters)
            temp_pars.readout_integration_delay = float(integration_delay)
            temp_pars.readout_integration_length = float(integration_length)
            temporary_parameters = {qubit_to_measure.uid: temp_pars}

            exp_workflow = iq_blobs.experiment_workflow(
                session=session, qpu=qpu, qubits=[qubit_to_measure.uid], states=states,
                options=options, temporary_parameters=temporary_parameters,
            )
            workflow_result = exp_workflow.run()

            shots = single_shot.collect_shots_from_result(workflow_result.output, qubit_to_measure.uid, states)
            shots_inner.append(np.array([shots[s] for s in states]))

            res = single_shot.analyze_single_shots(shots, state_0="g", state_1="e", plot=False)
            metrics_inner.append({k: v for k, v in res.items() if isinstance(v, (int, float, complex))})

            fidelities_inner.append(
                workflow_result.tasks["analysis_workflow"].output.get(qubit_to_measure.uid, np.nan)
            )
            matrices_inner.append(
                setup.get_analysis_task_output(workflow_result, "calculate_assignment_matrices").get(
                    qubit_to_measure.uid
                )
            )

        ss_delay_sweep_shots.append(shots_inner)
        ss_delay_sweep_metrics.append(metrics_inner)
        ss_delay_sweep_fidelities.append(fidelities_inner)
        ss_delay_sweep_matrices.append(matrices_inner)

ss_delay_sweep_arr = np.array(ss_delay_sweep_shots)
print("Sweep array shape (delays, integration lengths, states, shots):", ss_delay_sweep_arr.shape)

#### Analysis Plots

In [ ]:
fidelity_arr = np.array(ss_delay_sweep_fidelities, dtype=float)
rel_std_0_arr = np.array(
    [[m["rel_std_0"] for m in metrics_inner] for metrics_inner in ss_delay_sweep_metrics]
)

optimal_delay_index, optimal_length_index, optimal_fidelity = single_shot_analysis.pick_optimal_2d(fidelity_arr)
readout_integration_delay_opt = float(readout_integration_delay_arr[optimal_delay_index])
readout_integration_length_opt = float(readout_integration_length_arr[optimal_length_index])

print("Readout length is", readout_length)
print("Best assignment fidelity is", f"{optimal_fidelity * 100:0.2f} %",
      "for delay", np.round(readout_integration_delay_opt * 1e9, 3),
      "ns (index", optimal_delay_index, ") and integration length",
      readout_integration_length_opt, "(index", optimal_length_index, ").")

In [ ]:
fig, ax = single_shot_analysis.plot_fidelity_vs_2d_lines(
    readout_integration_delay_arr * 1e9, fidelity_arr, readout_integration_length_arr * 1e9,
    readout_integration_delay_opt * 1e9, "Readout integration delay, ns", qubit_to_measure.uid,
    legend_title="Integration length", series_fmt="{:.0f} ns",
)
fig.savefig(get_path_to_file("Single_shots_vs_delay_and_int_len_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
fig, ax = single_shot_analysis.plot_fidelity_heatmap(
    readout_integration_delay_arr * 1e9, readout_integration_length_arr * 1e6, fidelity_arr * 100,
    readout_integration_delay_opt * 1e9, readout_integration_length_opt * 1e6,
    "Readout integration delay, ns", "Readout integration length, us", qubit_to_measure.uid,
)
fig.savefig(get_path_to_file("Single_shots_delay_and_int_len_map_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
# set to a value != None to override the values extracted above.
manual_readout_integration_delay = None
manual_readout_integration_length = None
if manual_readout_integration_delay is not None:
    readout_integration_delay_opt = float(manual_readout_integration_delay)
if manual_readout_integration_length is not None:
    readout_integration_length_opt = float(manual_readout_integration_length)

if update_global_parameters:
    temp_pars = deepcopy(qubit_to_measure.parameters)
    temp_pars.readout_integration_delay = readout_integration_delay_opt
    temp_pars.readout_integration_length = readout_integration_length_opt
    qubit_to_measure.parameters = temp_pars
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("Readout integration delay: ", qubit_to_measure.parameters.readout_integration_delay)
print("Readout integration length: ", qubit_to_measure.parameters.readout_integration_length)

In [ ]:
data_ss = {
    "ss_delay_sweep_arr": ss_delay_sweep_arr,
    "states": list(states),
    "readout_integration_delay_sweep": readout_integration_delay_arr,
    "readout_integration_length_sweep": readout_integration_length_arr,
    "readout_length": readout_length,
    "assignment_fidelity": fidelity_arr,
    "rel_std_0": rel_std_0_arr,
    "optimal_delay_index": optimal_delay_index,
    "optimal_length_index": optimal_length_index,
    "n_shots": 2**n_avg_exponent,
    "comment": "Sweep integration delay and integration length. Optimal point maximizes assignment_fidelity (g/e/f).",
}
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_ss)
data_io.save_mat(get_path_to_file("Single_shots_0_pi_sweep_delay_and_ro_len_", ".mat"), payload)

best_matrix = ss_delay_sweep_matrices[optimal_delay_index][optimal_length_index]
single_shot_analysis.print_assignment_matrix(best_matrix)

## Single Shot Measurement for Different Readout Amplitudes and Lengths
*Optimum: the (readout amplitude, readout length) combination with the highest g/e/f assignment fidelity.*

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 16
states = "gef"

readout_amplitude_arr = np.linspace(0.05, 1, 20)
readout_length_arr = np.linspace(1900, 2000, 11) * 1e-9

qubit_to_measure = qubits[0]
readout_integration_delay = qubit_to_measure.parameters.readout_integration_delay

#### Run Workflow

In [ ]:
options = iq_blobs.experiment_workflow.options()
options.count(2**n_avg_exponent)
options.close_figures(True)
options.do_analysis(True)

ss_ro_sweep_metrics = []
ss_ro_sweep_fidelities = []
ss_ro_sweep_matrices = []

with setup.logging_disabled(folder_store, logging_store):
    for readout_amplitude in readout_amplitude_arr:
        print("Readout amplitude:", round(float(readout_amplitude), 5))

        metrics_inner, fidelities_inner, matrices_inner = [], [], []
        for readout_length in readout_length_arr:
            print("  Readout length:", round(readout_length * 1e9, 3), "ns")

            temp_pars = deepcopy(qubit_to_measure.parameters)
            temp_pars.readout_amplitude = float(readout_amplitude)
            temp_pars.readout_length = float(readout_length)
            temp_pars.readout_integration_length = float(readout_length)
            temp_pars.readout_integration_delay = readout_integration_delay
            temporary_parameters = {qubit_to_measure.uid: temp_pars}

            exp_workflow = iq_blobs.experiment_workflow(
                session=session, qpu=qpu, qubits=[qubit_to_measure.uid], states=states,
                options=options, temporary_parameters=temporary_parameters,
            )
            workflow_result = exp_workflow.run()

            shots = single_shot.collect_shots_from_result(workflow_result.output, qubit_to_measure.uid, states)
            res = single_shot.analyze_single_shots(shots, state_0="g", state_1="e", plot=False)
            metrics_inner.append({k: v for k, v in res.items() if isinstance(v, (int, float, complex))})

            fidelities_inner.append(
                workflow_result.tasks["analysis_workflow"].output.get(qubit_to_measure.uid, np.nan)
            )
            matrices_inner.append(
                setup.get_analysis_task_output(workflow_result, "calculate_assignment_matrices").get(
                    qubit_to_measure.uid
                )
            )

        ss_ro_sweep_metrics.append(metrics_inner)
        ss_ro_sweep_fidelities.append(fidelities_inner)
        ss_ro_sweep_matrices.append(matrices_inner)

#### Analysis Plots

In [ ]:
fidelity_arr = np.array(ss_ro_sweep_fidelities, dtype=float)
rel_std_0_arr = np.array(
    [[m["rel_std_0"] for m in metrics_inner] for metrics_inner in ss_ro_sweep_metrics]
)

optimal_amp_index, optimal_len_index, optimal_fidelity = single_shot_analysis.pick_optimal_2d(fidelity_arr)
readout_amplitude_opt = float(readout_amplitude_arr[optimal_amp_index])
readout_length_opt = float(readout_length_arr[optimal_len_index])

print("Best assignment fidelity is", f"{optimal_fidelity * 100:0.2f} %",
      "for amplitude", np.round(readout_amplitude_opt, 3),
      "(index", optimal_amp_index, ") and length", readout_length_opt, "(index", optimal_len_index, ").")

In [ ]:
fig, ax = single_shot_analysis.plot_fidelity_vs_2d_lines(
    readout_amplitude_arr, fidelity_arr, readout_length_arr * 1e9,
    readout_amplitude_opt, "Readout amplitude, a.u.", qubit_to_measure.uid,
    legend_title="Readout length", series_fmt="{:.0f} ns",
)
fig.savefig(get_path_to_file("Single_shots_vs_ro_amp_and_ro_len_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
fig, ax = single_shot_analysis.plot_fidelity_heatmap(
    readout_length_arr * 1e9, readout_amplitude_arr, fidelity_arr.T * 100,
    readout_length_opt * 1e9, readout_amplitude_opt,
    "Readout length, ns", "Readout amplitude, a.u.", qubit_to_measure.uid,
)
fig.savefig(get_path_to_file("Single_shots_ro_amp_ro_len_map_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
# set to a value != None to override the values extracted above.
manual_readout_amplitude = None
manual_readout_length = None
if manual_readout_amplitude is not None:
    readout_amplitude_opt = float(manual_readout_amplitude)
if manual_readout_length is not None:
    readout_length_opt = float(manual_readout_length)

if update_global_parameters:
    temp_pars = deepcopy(qubit_to_measure.parameters)
    temp_pars.readout_amplitude = readout_amplitude_opt
    temp_pars.readout_length = readout_length_opt
    temp_pars.readout_integration_length = readout_length_opt
    qubit_to_measure.parameters = temp_pars
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("Readout amplitude: ", qubit_to_measure.parameters.readout_amplitude)
print("Readout length: ", qubit_to_measure.parameters.readout_length)
print("Readout integration length: ", qubit_to_measure.parameters.readout_integration_length)

In [ ]:
data_ss = {
    "states": list(states),
    "readout_amplitude_sweep": readout_amplitude_arr,
    "readout_length_sweep": readout_length_arr,
    "readout_integration_delay": readout_integration_delay,
    "assignment_fidelity": fidelity_arr,
    "rel_std_0": rel_std_0_arr,
    "optimal_amp_index": optimal_amp_index,
    "optimal_len_index": optimal_len_index,
    "n_shots": 2**n_avg_exponent,
    "comment": "Sweep readout amplitude and readout length. Optimal point maximizes assignment_fidelity (g/e/f).",
}
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_ss)
data_io.save_mat(get_path_to_file("Single_shots_0_pi_sweep_ro_amp_and_ro_len_", ".mat"), payload)

best_matrix = ss_ro_sweep_matrices[optimal_amp_index][optimal_len_index]
single_shot_analysis.print_assignment_matrix(best_matrix)

In [ ]:
readout_opt = {
    "readout_amplitude": qubit_to_measure.parameters.readout_amplitude,
    "readout_length": qubit_to_measure.parameters.readout_length,
    "readout_pulse": qubit_to_measure.parameters.readout_pulse,
    "readout_integration_length": qubit_to_measure.parameters.readout_integration_length,
    "readout_integration_delay": qubit_to_measure.parameters.readout_integration_delay,
    "readout_integration_kernels_type": qubit_to_measure.parameters.readout_integration_kernels_type,
    "readout_resonator_frequency": qubit_to_measure.parameters.readout_resonator_frequency,
    "readout_lo_frequency": qubit_to_measure.parameters.readout_lo_frequency,
    "readout_range_out": qubit_to_measure.parameters.readout_range_out,
    "readout_range_in": qubit_to_measure.parameters.readout_range_in,
    "reset_delay_length": qubit_to_measure.parameters.reset_delay_length,
}
pprint(readout_opt)

# Population Measurements
Effective-temperature measurements of the qubit.

1. **Rabi Population Measurements (RPM)** — compare the ef-Rabi amplitude with and without a ge pre-pulse. The ratio of the two oscillation amplitudes gives the residual excited-state population `p_e` and thus the effective temperature. The ef pi-pulse used for the fast (two-point) version is calibrated in `Setup`.
2. **Three Level Population Measurements** — the six-sequence protocol (`x0`, `x1`, `x2`, `y0`, `y1`, `y2`) that yields the nine `A`/`B`/`C` estimators of `p_e/p_g` and the corresponding temperatures `TA1 ... TC3`, including the repeated (statistics) version, the projection-phase optimization and a variant with a ge pre-pulse of swept amplitude.
3. **Population Measurements for Different Readout Frequencies** — the same six-sequence protocol repeated for a range of readout resonator frequencies (slow!).

The readout parameters optimized above are the starting point here.

`laboneq_applications` has no population experiment, so the custom experiments/workflows live in `qubit_thermometry.workflow.experiments.{population,rabi_population,quick_rabi_population}` and their analysis in `qubit_thermometry.workflow.analysis.{population,rabi_population}`.

## Rabi Population Measurements (RPM)

### Setup

#### Oscillations Check

##### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 17

ef_amp_min, ef_amp_max, n_ef_amp = 0, 1, 51
ef_amplitudes = np.linspace(ef_amp_min, ef_amp_max, n_ef_amp)
ef_detuning = 0.0

qubit_to_measure = qubits[0]

temp_pars = deepcopy(qubit_to_measure.parameters)
temp_pars.resonance_frequency_ef = qubit_to_measure.parameters.resonance_frequency_ef + ef_detuning
temporary_parameters = {qubit_to_measure.uid: temp_pars}

print("ef drive length:            ", temp_pars.ef_drive_length)
print("ef drive pulse:             ", temp_pars.ef_drive_pulse)
print("ef resonance frequency:     ", temp_pars.resonance_frequency_ef * 1e-9, "GHz")
print("ge drive amplitude pi:      ", temp_pars.ge_drive_amplitude_pi)
print("Reset delay length:         ", temp_pars.reset_delay_length)
print("Readout amplitude:          ", temp_pars.readout_amplitude)
print("Number of averages:         ", 2**n_avg_exponent)

##### Run Workflows

In [ ]:
options = rabi_population_experiment_workflow.options()
options.count(2**n_avg_exponent)

exp_workflow = rabi_population_experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    ef_amplitudes=ef_amplitudes,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

##### Analysis Plots

In [ ]:
rpm_res = rabi_population_analysis.collect_rpm_traces(
    workflow_result.output, qubit_to_measure.uid, ef_amplitudes
)
rpm_popt, rpm_pcov = rabi_population_analysis.fit_rpm_oscillations(rpm_res, freq_estimate=8)

In [ ]:
fig, axs, rpm_amp_plot = rabi_population_analysis.plot_rpm_oscillations(rpm_res, rpm_popt, qubit_to_measure.uid)
fig.savefig(get_path_to_file("RPM_full_amp_sweep_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Pi Pulse Check

##### Analysis Plots

In [ ]:
# alternatives, e.g. amp_with = rpm_amp_plot[np.argmax(func_osc(rpm_amp_plot, *fit_params_with))]
amp_with = 0.458069
amp_wo = amp_with

fig, axs, amp_avg = rabi_population_analysis.plot_pi_pulse_check(
    rpm_res, rpm_popt, rpm_amp_plot, amp_with, amp_wo, qubit_to_measure.uid, signal_in_mag=True,
)
fig.savefig(get_path_to_file("RPM_x180_ef_calibration_", ".png"), dpi=600, format="png", bbox_inches="tight")

##### Update Parameters

In [ ]:
# set to a value != None to override the value extracted above.
manual_ef_drive_amplitude_pi = None
ef_drive_amplitude_pi_rpm = float(amp_avg)
if manual_ef_drive_amplitude_pi is not None:
    ef_drive_amplitude_pi_rpm = float(manual_ef_drive_amplitude_pi)

temporary_parameters[qubit_to_measure.uid].ef_drive_amplitude_pi = ef_drive_amplitude_pi_rpm

if update_global_parameters:
    qubit_to_measure.parameters = deepcopy(temporary_parameters[qubit_to_measure.uid])
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("RPM ef drive amplitude pi: ", ef_drive_amplitude_pi_rpm)
print("ef drive amplitude pi (QPU): ", qubit_to_measure.parameters.ef_drive_amplitude_pi)

### Population Measurement

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 17
repetitions = 20

qubit_to_measure = qubits[0]

# the RPM pi-pulse calibrated above; the rest of the RPM parameters are unchanged.
temp_pars = deepcopy(qubit_to_measure.parameters)
temp_pars.ef_drive_amplitude_pi = ef_drive_amplitude_pi_rpm
temporary_parameters = {qubit_to_measure.uid: temp_pars}

print("ef drive amplitude pi: ", temp_pars.ef_drive_amplitude_pi)
print("Number of averages:    ", 2**n_avg_exponent)
print("Repetitions:           ", repetitions)

#### Run Workflow

In [ ]:
options = quick_rabi_population_experiment_workflow.options()
options.count(2**n_avg_exponent)

exp_workflow = quick_rabi_population_experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    repetitions=repetitions,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
rpm_res = rabi_population_analysis.collect_quick_rpm_traces(workflow_result.output, qubit_to_measure.uid)
qrpm_dict = temperature.qrpm_dict_from_results([rpm_res])

In [ ]:
rabi_population_analysis.plot_quick_rpm_iq(rpm_res, qubit_to_measure.uid)

In [ ]:
fig, axs = rabi_population_analysis.plot_quick_rpm_points(
    rpm_res, rpm_popt, rpm_amp_plot, ef_drive_amplitude_pi_rpm, qubit_to_measure.uid
)
fig.savefig(get_path_to_file("Quick_RPM_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
iq_amp_with, iq_amp_wo = rabi_population_analysis.compute_iq_pulse_distances(rpm_res)
rabi_population_analysis.plot_iq_pulse_distances(iq_amp_with, iq_amp_wo, qubit_to_measure.uid)

### Temperature

#### Analysis Plots

In [ ]:
f_q, anharm = temperature.get_qubit_temperature_parameters(qubit_to_measure)
print("f_q:   ", f_q, "GHz")
print("anharm:", anharm, "GHz")

rpm_pe = iq_amp_wo / (iq_amp_wo + iq_amp_with)
rpm_temp = temperature.T_calc(C=rpm_pe / (1 - rpm_pe), f_q=f_q)

fig_series, ax_series, fig_hist, ax_hist = rabi_population_analysis.plot_rpm_temperature(
    rpm_pe, rpm_temp, qubit_to_measure.uid
)
fig_series.savefig(get_path_to_file("Quick_RPM_temperature_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_rpm = {
    "ef_amplitude_sweep": ef_amplitudes,
    "rpm_mag_with": rpm_res["with"]["mag_max"],
    "rpm_mag_wo": rpm_res["wo"]["mag_max"],
    "iq_amp_with": iq_amp_with,
    "iq_amp_wo": iq_amp_wo,
    "rpm_pe": rpm_pe,
    "rpm_temp": rpm_temp,
    "ef_drive_amplitude_pi_rpm": ef_drive_amplitude_pi_rpm,
    "n_shots": 2**n_avg_exponent,
    "repetitions": repetitions,
    "f_q": f_q,
    "anharm": anharm,
    "comment": "Quick Rabi population measurement, temperature in K",
}
data_rpm.update(qrpm_dict)
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_rpm)
data_io.save_mat(get_path_to_file("RPM_temperature_", ".mat"), payload)

## Three Level Population Measurements

### Single-Point Experiment

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 16

qubit_to_measure = qubits[0]
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters = {qubit_to_measure.uid: temp_pars}

print("ge drive amplitude pi:      ", temp_pars.ge_drive_amplitude_pi)
print("ef drive amplitude pi:      ", temp_pars.ef_drive_amplitude_pi)
print("Readout amplitude:          ", temp_pars.readout_amplitude)
print("Readout length:             ", temp_pars.readout_length)
print("Readout resonator frequency:", temp_pars.readout_resonator_frequency)
print("Reset delay length:         ", temp_pars.reset_delay_length)
print("Number of averages:         ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = population_experiment_workflow.options()
options.count(2**n_avg_exponent)

exp_workflow = population_experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
pop_result = temperature.collect_population_from_result(workflow_result.output, qubit_to_measure.uid)
for label in temperature.pop_label_list:
    print(f"{label}: {pop_result[label]}")

### Population Statistics

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 16
n_measurements = 3

# estimators that are excluded from the plots (e.g. skip = ['TC2'])
skip = []
three_levels = True

qubit_to_measure = qubits[0]
temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters = {qubit_to_measure.uid: temp_pars}

f_q, anharm = temperature.get_qubit_temperature_parameters(qubit_to_measure)
print("f_q:               ", f_q, "GHz")
print("anharm:            ", anharm, "GHz")
print("Number of averages:", 2**n_avg_exponent)
print("Number of runs:    ", n_measurements)

#### Run Workflow

In [ ]:
options = population_experiment_workflow.options()
options.count(2**n_avg_exponent)

pop_full_results = {}

with setup.logging_disabled(folder_store, logging_store):
    for i in range(n_measurements):
        print("Iteration: ", i)

        exp_workflow = population_experiment_workflow(
            session=session, qpu=qpu, qubit=qubit_to_measure.uid,
            options=options, temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()

        pop_result = temperature.collect_population_from_result(workflow_result.output, qubit_to_measure.uid)
        for label in temperature.pop_label_list:
            pop_full_results.setdefault(label, []).append(pop_result[label])

for label in temperature.pop_label_list:
    pop_full_results[label] = np.array(pop_full_results[label])

print("Population array shape (runs,):", pop_full_results["x0"].shape)

#### Analysis Plots

In [ ]:
# fast temperature extraction (projection onto I and Q at phase 0)
TABC_I, TABC_Q = temperature.make_all_temperatures(pop_full_results, f_q, anharm, three_levels=three_levels)

fig, ax = plotting.plot_temperatures(TABC_I, TABC_Q, skip=skip)
fig.savefig(get_path_to_file("Population_stat_", ".png"), dpi=600, format="png", bbox_inches="tight")

STAT_I = temperature.get_stat(TABC_I)
STAT_Q = temperature.get_stat(TABC_Q)

# give plot_stat copies so it cannot modify the original statistics
fig, ax = plotting.plot_stat(
    {k: v.copy() for k, v in STAT_I.items()}, {k: v.copy() for k, v in STAT_Q.items()}
)

for k in STAT_I:
    print(k, "| I mean:", np.round(STAT_I[k][0], 3), "rel. err:", np.round(STAT_I[k][2], 4),
          "| Q mean:", np.round(STAT_Q[k][0], 3), "rel. err:", np.round(STAT_Q[k][2], 4))

fig.savefig(get_path_to_file("Population_stat_bars_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
# population, 3-level method, proper rotation ('parallel' approach)
ABC = temperature.get_ABC_parallel(pop_full_results)
TABC = temperature.get_temperature(ABC, f_q, anharm, three_levels=three_levels)

fig, ax = plotting.plot_temp_single(TABC, skip=skip)

STAT = temperature.get_stat(TABC)
fig_stat, ax_stat = plotting.plot_stat(STAT, STAT)

fig.savefig(get_path_to_file("Population_stat_parallel_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
population_analysis.plot_iq_projection(
    pop_full_results, temperature.pop_label_list, qubit_to_measure.uid, "Population measurements on the IQ plane"
)

In [ ]:
# specific A1/B2/C3 estimator combination, for a quick pe/pg and temperature readout
fig_pe, ax_pe, fig_temp, ax_temp, estimators = population_analysis.plot_population_estimators_a1_b2_c3(
    pop_full_results, f_q, qubit_to_measure.uid
)
fig_temp.savefig(get_path_to_file("Population_stat_temperatures_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_t_stat = {f"pop_{label}": pop_full_results[label] for label in temperature.pop_label_list}
data_t_stat.update({f"{k}_I": v for k, v in TABC_I.items()})
data_t_stat.update({f"{k}_Q": v for k, v in TABC_Q.items()})
data_t_stat.update({f"{k}_parallel": v for k, v in TABC.items()})
data_t_stat.update({k: v for k, v in estimators.items()})
data_t_stat.update({
    "n_measurements": n_measurements,
    "n_shots": 2**n_avg_exponent,
    "f_q": f_q,
    "anharm": anharm,
    "three_levels": three_levels,
    "comment": "Three level population statistics; temperatures in mK",
})
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_t_stat)
data_io.save_mat(get_path_to_file("Teff_stat_all_unsh_", ".mat"), payload)

### Rotation and Measurement Optimisation

#### Experiment Parameters

In [ ]:
# No new experiment is run here: the projection phase of the data acquired in
# `Population Statistics` is optimized.
phase_arr = np.linspace(-1, 1, 65) * np.pi / 2

# estimators that are excluded from the optimization (e.g. skip = ['TA1', 'TA3'])
skip_optimization = []
pop_data = pop_full_results

#### Analysis Plots

In [ ]:
STAT_I, STAT_Q = temperature.make_rotation_temperature(pop_data, phase_arr, f_q, anharm, three_levels=three_levels)

for info_type in ["mean", "error", "rel_err"]:
    fig, ax = plotting.plot_rotation_stat(phase_arr, STAT_I, STAT_Q, info_type=info_type)
    fig.savefig(get_path_to_file(f"Population_rotation_{info_type}_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
for k in STAT_I:
    print("For", k)
    print("  I: min rel. err:", np.nanmin(STAT_I[k][:, 2]), "at phase point", np.nanargmin(STAT_I[k][:, 2]))
    print("  Q: min rel. err:", np.nanmin(STAT_Q[k][:, 2]), "at phase point", np.nanargmin(STAT_Q[k][:, 2]))

In [ ]:
optimal_method = temperature.make_optimal_temperature(
    pop_data, phase_arr, f_q, anharm, skip=skip_optimization, three_levels=three_levels
)
pprint(optimal_method)

Teff = temperature.find_optimal_temperature(pop_data, optimal_method, f_q, anharm, three_levels=three_levels)

fig, ax = plt.subplots()
ax.plot(Teff, "o-")
ax.set_xlabel("Experiment number")
ax.set_ylabel("T_eff, mK")
ax.set_title(f"Optimal method {optimal_method['method']} ({optimal_method['axes']}) - {qubit_to_measure.uid}")

print("Mean optimal temperature:", np.nanmean(Teff), "mK")

fig.savefig(get_path_to_file("Population_optimal_temperature_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
# statistics at a single (zero) phase, for comparison
STAT_I_0, STAT_Q_0 = temperature.make_rotation_temperature(pop_data, [0], f_q, anharm, three_levels=three_levels)
fig, ax = plotting.plot_stat(
    {k: v.copy() for k, v in STAT_I_0.items()}, {k: v.copy() for k, v in STAT_Q_0.items()}
)

#### Update Parameters

In [ ]:
data_t_opt = {
    "phase_arr": phase_arr,
    "optimal_phase": optimal_method["phase"],
    "optimal_method": optimal_method["method"],
    "optimal_axes": optimal_method["axes"],
    "optimal_rel_err": optimal_method["rel_err"],
    "Teff": Teff,
    "f_q": f_q,
    "anharm": anharm,
    "comment": "Projection-phase optimization of the population statistics",
}
data_t_opt.update({f"STAT_{k}_I": v for k, v in STAT_I.items()})
data_t_opt.update({f"STAT_{k}_Q": v for k, v in STAT_Q.items()})
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_t_opt)
data_io.save_mat(get_path_to_file("Teff_rotation_optimization_", ".mat"), payload)

### Population with Prepulse

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 16

qubit_to_measure = qubits[0]
prepulse_amp_min = 0
prepulse_amp_max = qubit_to_measure.parameters.ge_drive_amplitude_pi
n_prepulse_amp = 51
prepulse_amplitudes = np.linspace(prepulse_amp_min, prepulse_amp_max, n_prepulse_amp)

temp_pars = deepcopy(qubit_to_measure.parameters)
temporary_parameters = {qubit_to_measure.uid: temp_pars}

print("ge drive pulse:        ", temp_pars.ge_drive_pulse)
print("ge drive length:       ", temp_pars.ge_drive_length)
print("ge drive amplitude pi: ", temp_pars.ge_drive_amplitude_pi)
print("Number of averages:    ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = population_experiment_workflow.options()
options.count(2**n_avg_exponent)

exp_workflow = population_experiment_workflow(
    session=session,
    qpu=qpu,
    qubit=qubit_to_measure.uid,
    prepulse_amplitudes=prepulse_amplitudes,
    options=options,
    temporary_parameters=temporary_parameters,
)
workflow_result = exp_workflow.run()

#### Analysis Plots

In [ ]:
pulse_amp = prepulse_amplitudes
pop_prepulse_results = temperature.collect_population_from_result(workflow_result.output, qubit_to_measure.uid)

population_analysis.plot_prepulse_normalized_x0(pulse_amp, pop_prepulse_results["x0"], qubit_to_measure.uid)

In [ ]:
fig, ax, pe0 = population_analysis.compute_and_plot_prepulse_estimator(
    pulse_amp, pop_prepulse_results, qubit_to_measure.uid
)
fig.savefig(get_path_to_file("Population_prepulse_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_pop_prep = {f"pop_{label}": pop_prepulse_results[label] for label in temperature.pop_label_list}
data_pop_prep.update({
    "prepulse_amplitude_sweep": pulse_amp,
    "pe0": pe0,
    "n_shots": 2**n_avg_exponent,
    "comment": "Three level population vs. ge pre-pulse amplitude",
})
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_pop_prep)
data_io.save_mat(get_path_to_file("Population_prep_", ".mat"), payload)

## Population Measurements for Different Readout Frequencies
The three-level protocol repeated for a range of readout resonator frequencies. Every frequency needs its own compilation, so this is slow.

#### Experiment Parameters

In [ ]:
update_global_parameters = False

n_avg_exponent = 16

freq_range = 5e6
n_points = 51

qubit_to_measure = qubits[0]
readout_resonator_frequency_arr = (
    qubit_to_measure.parameters.readout_resonator_frequency
    + np.linspace(-freq_range / 2, freq_range / 2, n_points)
)

readout_range_out = qubit_to_measure.parameters.readout_range_out
readout_integration_delay = qubit_to_measure.parameters.readout_integration_delay

print("Readout resonator frequencies: ",
      readout_resonator_frequency_arr[0] * 1e-9, "...", readout_resonator_frequency_arr[-1] * 1e-9, "GHz")
print("Number of averages:            ", 2**n_avg_exponent)

#### Run Workflow

In [ ]:
options = population_experiment_workflow.options()
options.count(2**n_avg_exponent)

pop_i_results = {}

with setup.logging_disabled(folder_store, logging_store):
    for readout_resonator_frequency in readout_resonator_frequency_arr:
        print("Readout resonator frequency:", round(readout_resonator_frequency * 1e-9, 6), "GHz")

        temp_pars = deepcopy(qubit_to_measure.parameters)
        temp_pars.readout_resonator_frequency = float(readout_resonator_frequency)
        temp_pars.readout_range_out = readout_range_out
        temp_pars.readout_integration_delay = readout_integration_delay
        temporary_parameters = {qubit_to_measure.uid: temp_pars}

        exp_workflow = population_experiment_workflow(
            session=session, qpu=qpu, qubit=qubit_to_measure.uid,
            options=options, temporary_parameters=temporary_parameters,
        )
        workflow_result = exp_workflow.run()

        pop_result = temperature.collect_population_from_result(workflow_result.output, qubit_to_measure.uid)
        for label in temperature.pop_label_list:
            pop_i_results.setdefault(label, []).append(pop_result[label])

for label in temperature.pop_label_list:
    pop_i_results[label] = np.array(pop_i_results[label])

print("Population array shape (frequencies,):", pop_i_results["x0"].shape)

#### Analysis Plots

In [ ]:
fig_iq, ax_iq, fig, ax = population_analysis.plot_population_vs_readout_frequency(
    pop_i_results, readout_resonator_frequency_arr, qubit_to_measure.uid
)
fig.savefig(get_path_to_file("Population_vs_ro_freq_", ".png"), dpi=600, format="png", bbox_inches="tight")

In [ ]:
# fast temperature extraction per readout frequency
skip = []
TABC_I, TABC_Q = temperature.make_all_temperatures(pop_i_results, f_q, anharm, three_levels=three_levels)

fig, ax = plotting.plot_temperatures(TABC_I, TABC_Q, skip=skip)

fig_i, ax_i = population_analysis.plot_temperature_vs_readout_frequency(
    TABC_I, TABC_Q, readout_resonator_frequency_arr, qubit_to_measure.uid
)
population_analysis.plot_temperature_iq_correlation(TABC_I, TABC_Q, qubit_to_measure.uid)

STAT_I = temperature.get_stat_nan(TABC_I)
STAT_Q = temperature.get_stat_nan(TABC_Q)
fig_stat, ax_stat = plotting.plot_stat(STAT_I, STAT_Q)

fig_i.savefig(get_path_to_file("Population_vs_ro_freq_temperature_", ".png"), dpi=600, format="png", bbox_inches="tight")

#### Update Parameters

In [ ]:
data_pop_ro_freq = {f"pop_{label}": pop_i_results[label] for label in temperature.pop_label_list}
data_pop_ro_freq.update({f"{k}_I": v for k, v in TABC_I.items()})
data_pop_ro_freq.update({f"{k}_Q": v for k, v in TABC_Q.items()})
data_pop_ro_freq.update({
    "readout_resonator_frequency_sweep": readout_resonator_frequency_arr,
    "readout_range_out": readout_range_out,
    "readout_integration_delay": readout_integration_delay,
    "n_shots": 2**n_avg_exponent,
    "f_q": f_q,
    "anharm": anharm,
    "comment": "Three level population vs. readout resonator frequency; temperatures in mK",
})
payload = data_io.build_mat_payload(qubit_to_measure, sample_name, qubit_name, cooldown_start_date, data=data_pop_ro_freq)
data_io.save_mat(get_path_to_file("Population_vs_ro_freq_", ".mat"), payload)

In [ ]:
# set to a value != None to move the readout to the frequency with the lowest
# temperature spread (old notebook did this by hand).
manual_readout_resonator_frequency = None

optimal_index, readout_resonator_frequency_opt, rel_err_vs_freq = population_analysis.pick_optimal_readout_frequency(
    TABC_I, readout_resonator_frequency_arr, skip=skip
)
if manual_readout_resonator_frequency is not None:
    readout_resonator_frequency_opt = float(manual_readout_resonator_frequency)

print("Optimal readout resonator frequency:",
      round(readout_resonator_frequency_opt * 1e-9, 6), "GHz", "(index", optimal_index, ")")

if update_global_parameters:
    temp_pars = deepcopy(qubit_to_measure.parameters)
    temp_pars.readout_resonator_frequency = readout_resonator_frequency_opt
    qubit_to_measure.parameters = temp_pars
    qpu_io.save_qpu(qpu, qpu_file_path)
    print("Updated global experiment parameters!\n")

print("Readout resonator frequency: ", qubit_to_measure.parameters.readout_resonator_frequency * 1e-9, " GHz")